# ⚡ Flux de travail concurrent des agents avec les modèles GitHub (Python)

## 📋 Tutoriel avancé de traitement parallèle

Ce notebook démontre les **modèles de flux de travail concurrentiels** en utilisant le Microsoft Agent Framework. Vous apprendrez à construire des flux de travail à hautes performances et traitement parallèle où plusieurs agents IA s’exécutent simultanément, améliorant considérablement le débit et permettant des processus métier multi-thread sophistiqués.

## 🎯 Objectifs d'apprentissage

### 🚀 **Fondamentaux du traitement concurrent**
- **Exécution parallèle des agents** : Exécuter plusieurs agents simultanément pour une efficacité maximale
- **Orchestration du flux de travail** : Coordonner les opérations concurrentes tout en maintenant la cohérence des données
- **Optimisation des performances** : Obtenir une accélération significative grâce au traitement parallèle
- **Gestion des ressources** : Utiliser efficacement les ressources des modèles IA à travers les opérations concurrentes

### 🏗️ **Modèles avancés de concurrence**
- **Traitement Fork-Join** : Répartir le travail entre plusieurs agents et fusionner les résultats
- **Parallélisme en pipeline** : Chevauchement des étapes d’exécution pour un débit continu
- **Répartition de charge** : Distribuer le travail uniformément parmi les ressources agents disponibles
- **Points de synchronisation** : Coordonner les agents concurrents aux étapes critiques du flux de travail

### 🏢 **Applications concurrentes en entreprise**
- **Traitement de documents à volume élevé** : Traiter plusieurs documents simultanément
- **Analyse de contenu en temps réel** : Analyse concurrente des flux de données entrants
- **Optimisation du traitement par lots** : Maximiser le débit pour des opérations à grande échelle
- **Analyse multi-modale** : Traitement parallèle de différents types de contenu (texte, images, données)

## ⚙️ Prérequis & configuration

### 📦 **Dépendances requises**

Installer Agent Framework avec les capacités de flux de travail concurrent :

```bash
pip install agent-framework-core -U
```

### 🔑 **Configuration des modèles GitHub**

**Configuration de l’environnement (.env) :**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**Considérations pour le traitement concurrent :**
- **Limites de taux** : Surveiller les limites d’appels API des modèles GitHub pour les requêtes concurrentes
- **Utilisation des ressources** : Prendre en compte la mémoire et l’utilisation CPU avec plusieurs agents concurrents
- **Gestion des erreurs** : Implémenter une récupération robuste des erreurs pour les opérations parallèles

### 🏗️ **Architecture du flux de travail concurrent**

```mermaid
graph TD
    A[Démarrage du flux de travail] --> B[Exécution concurrente]
    B --> C[Piscine d'agents 1]
    B --> D[Piscine d'agents 2]
    B --> E[Piscine d'agents 3]
    C --> F[Agrégation des résultats]
    D --> F
    E --> F
    F --> G[Sortie finale]
    
    H[API des modèles GitHub] --> C
    H --> D
    H --> E
```

**Bénéfices clés :**
- **⚡ Performance** : Accélération significative grâce à l’exécution parallèle
- **📈 Scalabilité** : Gérer des charges accrues sans augmentation proportionnelle du temps
- **🔄 Efficacité** : Meilleure utilisation des ressources computationnelles disponibles
- **🎯 Débit** : Traiter plus de travail dans le même laps de temps

## 🎨 **Modèles de conception pour flux de travail concurrent**

### 🔍 **Pipeline de recherche & analyse**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Flux de traitement des données**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Pipeline de création de contenu**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Traitement multi-étapes**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Avantages de performance en entreprise**

### ⚡ **Optimisation du débit**
- **Exécution parallèle** : Plusieurs agents travaillant simultanément
- **Utilisation des ressources** : Efficacité maximale de la capacité disponible du modèle IA
- **Réduction du temps** : Diminution significative du temps total de traitement
- **Architecture évolutive** : Ajouter facilement plus d’agents concurrents selon les besoins

### 🛡️ **Fiabilité & résilience**
- **Tolérance aux pannes** : Les défaillances d’agents individuels ne stoppent pas tout le flux
- **Isolation des erreurs** : Les problèmes dans une branche concurrente n'affectent pas les autres
- **Dégradation progressive** : Le système continue à fonctionner même avec une capacité d’agents réduite
- **Mécanismes de récupération** : Retentatives automatiques et gestion d’erreurs pour les opérations échouées

### 📊 **Suivi & observabilité**
- **Suivi de l’exécution concurrente** : Surveiller la progression de toutes les opérations parallèles
- **Mesures de performance** : Mesurer l’accélération et les gains d’efficacité
- **Analyse de l’utilisation des ressources** : Optimiser l’allocation des agents concurrents
- **Identification des goulots d’étranglement** : Trouver et résoudre les contraintes de performance

Construisons des flux de travail IA concurrents à haute performance ! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
